In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


print("PyTorch version:", torch.__version__)
print("Cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Cuda Memory Size:", round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2), "GB")


def log_sum_exp(logits_cut_max: torch.Tensor):
    return torch.log(torch.sum(torch.exp(logits_cut_max), dim=-1, keepdim=True))


def log_softmax(logits: torch.Tensor):
    max_val = torch.max(logits, dim=-1, keepdim=True).values
    return (logits - max_val) - log_sum_exp(logits - max_val)


def softmax(logits: torch.Tensor):
    return torch.exp(log_softmax(logits))


def get_ce_loss(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor):
    """Token-mean CE: 每个 token 等权;长样本权重更大"""
    logps = log_softmax(logits)
    nll = -torch.gather(logps, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)   # −log p ≥ 0
    loss = (nll * mask).sum() / mask.sum().clamp_min(1.0)
    return loss


def get_square_ce_loss(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor):
    """
    sqrt-N CE (length-balanced):
        每个 token 权重 = 1 / sqrt(N_i),N_i 是该样本的有效 token 数
        => 每个样本对 loss 的贡献权重 = sqrt(N_i)
        介于 token-mean (权重 ∝ N_i) 和 sample-mean (权重 = 1) 之间
        常用于 NLP 中避免长序列淹没短序列的"balanced loss"
    """
    logps = log_softmax(logits)
    nll = -torch.gather(logps, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)

    N = mask.sum(dim=-1, keepdim=True).clamp_min(1.0)        # [B, 1]
    weight = mask / torch.sqrt(N)                            # [B, T]
    loss = (nll * weight).sum() / weight.sum().clamp_min(1e-8)
    return loss


def get_logits(model: nn.Module, input_ids: torch.Tensor):
    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits
    return logits


if __name__ == "__main__":
    input_ids = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
    targets = torch.tensor([[2, 3, 4, 5], [6, 7, 8, 9]])
    attention_mask = torch.tensor([[1, 1, 1, 0], [1, 1, 0, 0]], dtype=torch.float)

    class DummyModel(nn.Module):
        def __init__(self):
            super().__init__()

        def forward(self, input_ids):
            batch_size, seq_len = input_ids.shape
            vocab_size = 10
            logits = torch.randn(batch_size, seq_len, vocab_size)
            return type("Output", (object,), {"logits": logits})()

    torch.manual_seed(42)
    model = DummyModel()
    logits = get_logits(model, input_ids)
    ce_loss = get_ce_loss(logits, targets, attention_mask)
    square_ce_loss = get_square_ce_loss(logits, targets, attention_mask)

    print(f"Cross-Entropy Loss:       {ce_loss:.4f}")    # 应为正数
    print(f"Square (sqrt-N) CE Loss:  {square_ce_loss:.4f}")


### Lora

In [ ]:
import torch
import torch.nn as nn


class LoraLinear(nn.Module):
    """
    LoRA: W = W₀ + (α/r) · B @ A
    - W₀ 冻结,A/B 可训练
    - 标准 init:A 用 Kaiming/Bessel uniform,B 用 zeros
      → 训练初始 delta = 0,等价于 frozen W₀
    """
    def __init__(self, in_features: int, out_features: int, r: int = 4, alpha: float = 1.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # 原始冻结层(注意命名:self.frozen 而不是误导性的 self.weight)
        self.frozen = nn.Linear(in_features, out_features, bias=False)
        # for p in self.frozen.parameters():
        #     p.requires_grad = False                    # 冻结 weight (无 bias 所以不用单独处理)
        self.frozen.weight.requires_grad = False

        # LoRA 低秩增量
        self.lora_A = nn.Linear(in_features, r, bias=False)
        self.lora_B = nn.Linear(r, out_features, bias=False)
        # 标准 LoRA 初始化:A 用 Kaiming uniform,B 用 zeros
        nn.init.kaiming_uniform_(self.lora_A.weight, a=5 ** 0.5)
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        # 等价于 W₀ @ x + B @ (A @ x) * scaling
        return self.frozen(x) + self.lora_B(self.lora_A(x)) * self.scaling


# ============ 验证 ============
torch.manual_seed(0)
layer = LoraLinear(in_features=8, out_features=4, r=2, alpha=4.0)

# 1. 初始 delta 应该为 0(因为 B 初始化为 zeros)
x = torch.randn(5, 8)
delta = (layer(x) - layer.frozen(x)).abs().max().item()
assert delta < 1e-7, f"初始 LoRA 增量应该为 0,实际 {delta}"
print(f"✅ 初始 delta = {delta:.2e} (B=zeros 保证训练起点等价于 frozen)")




# RLs

### DPO

$r_c = log(pi_theta_c) - log(pi_old_c)$

r_w = log(pi_theta_w) - log(pi_old_w)

L_{dpo} = -E[log_sigmoid(beta * (r_c - r_w))] 

In [ ]:
def get_logps(logits: torch.Tensor, targets: torch.Tensor, mask: torch.Tensor, sum: bool = False):
    logps = log_softmax(logits)
    logps_target = torch.gather(logps, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)  # [B, T]
    logps_target_masked = logps_target * mask.float()  # [B, T]
    return logps_target_masked.sum(dim=-1) if sum else logps_target_masked  # [B] or [B, T]
    

def dpo_loss(ref_logits: torch.Tensor, 
             chosen_logits: torch.Tensor, 
             reject_logits: torch.Tensor, 
             targets: torch.Tensor, 
             mask: torch.Tensor, 
             beta: float = 1.0):
    """
    DPO loss (Direct Preference Optimization)
    - ref_logits: [B, T, V] 参考模型的 logits
    - chosen_logits: [B, T, V] 选择的样本的 logits
    - reject_logits: [B, T, V] 拒绝的样本的 logits
    - targets: [B, T] 目标 token 的索引
    - mask: [B, T] 用于掩码的张量
    - beta: 温度参数
    """
    ref_logps = get_logps(ref_logits, targets, mask)  # [B] 
    chosen_logps = get_logps(chosen_logits, targets, mask)  # [B]
    reject_logps = get_logps(reject_logits, targets, mask)  #[B]

    log_ratio = (chosen_logps - ref_logps) - (reject_logps - ref_logps)      # [B]
    dpo_loss = -torch.log(torch.sigmoid(beta * log_ratio)).mean()  # [B]

    return dpo_loss

### GRPO

$\mathcal{L}_{GRPO} = -\mathbb{E}_{q \sim P(Q), \{o_i\}_{i=1}^G \sim \pi_{\theta_{old}}} \left[ \frac{1}{G} \sum_{i=1}^G \left( \min\left(\rho_i \hat{A}_i, \text{clip}(\rho_i, 1-\epsilon, 1+\epsilon) \hat{A}_i\right) - \beta \mathbb{D}_{KL}(\pi_\theta \| \pi_{ref}) \right) \right]$




In [ ]:
def grpo_loss(new_logits: torch.Tensor,
              old_logits: torch.Tensor,
              ref_logits: torch.Tensor,
              mask: torch.Tensor,
              targets: torch.Tensor,
              rewards: torch.Tensor,
              beta: float = 0.01,
              epsilon: float = 0.2
              ):
    bsz, rollout_n, seq_len, _ = new_logits.shape
    new_logpbs = get_logps(new_logits, targets, mask)  # [B, N, T]
    old_logpbs = get_logps(old_logits, targets, mask)  # [B, N, T]
    ref_logpbs = get_logps(ref_logits, targets, mask) # [B, N, T]
    rewards = (rewards - rewards.mean(dim=1, keepdim=True)) / (rewards.std(dim=1, keepdim=True) + 1e-8) # [B, N]
    adv = (rewards / mask.sum(dim=-1)).unsqueeze(-1).expand(bsz, rollout_n, seq_len)  # [B, N, T]
    ratio = torch.exp(new_logpbs - old_logpbs)  # [B, N, T]
    kl_ratio = new_logpbs - ref_logpbs  # [B, N, T]
    kl_div = torch.exp(kl_ratio) - kl_ratio - 1
    
    loss = torch.min(ratio * adv, torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * adv) - beta * kl_div
    loss = -(loss * mask.float()).sum() / mask.sum().clamp_min(1e-8)
    return loss

# Test grpo loss
if __name__ == "__main__":
    torch.manual_seed(42)
    bsz, rollout_n, seq_len, vocab_size = 2, 3, 4, 5
    new_logits = torch.randn(bsz, rollout_n, seq_len, vocab_size)
    old_logits = torch.randn(bsz, rollout_n, seq_len, vocab_size)
    ref_logits = torch.randn(bsz, rollout_n, seq_len, vocab_size)
    targets = torch.randint(0, vocab_size, (bsz, rollout_n, seq_len))
    mask = torch.randint(0, 2, (bsz, rollout_n, seq_len)).float()
    rewards = torch.randn(bsz, rollout_n)
    
    loss = grpo_loss(new_logits, old_logits, ref_logits, mask, targets, rewards)
    print(f"GRPO Loss: {loss.item():.4f}")
    
    

### PPO

In [ ]:
# def ppo_loss(cur_logits: torch.Tensor,
#              ref_logits: torch.Tensor,
#              old_log_probs: torch.Tensor,
#              values: torch.Tensor,
#              rewards: torch.Tensor,
#              mask: torch.Tensor,
#              beta: float = 0.01,
#              epsilon: float = 0.2
#              lambda: float = 0.95
#              gamma: float = 0.99
#              )

class PPO:
    def __init__(self, beta: float = 0.01, epsilon: float = 0.2, lambda_: float = 0.95, gamma: float = 0.99):
        self.beta = beta
        self.epsilon = epsilon
        self.lambda_ = lambda_
        self.gamma = gamma

    def compute_gae(self, rewards: torch.Tensor, values: torch.Tensor, mask: torch.Tensor):
        B, T = values.shape
        gae = 0.0
        delta, adv = torch.zeros_like(values), torch.zeros_like(values)
        for t in reversed(range(T)):
            last_value = 0.0 if t == T - 1 else values[:, -1]
            delta[:, t] = rewards[:, t] + self.gamma * last_value - values[:, t]
            gae = gae + self.gamma * self.lambda_ * delta[:, t]
            adv[:, t] = gae
        returns = adv + values
            
            
        return adv, returns
    
    def critic_loss(self, returns, values, mask):
        mse_loss = (returns - values) ** 2
        loss = (mse_loss * mask.float()).sum() / mask.sum().clamp(min=1.0)
        return loss
    
    
    def get_ppo_loss(self, cur_logps, ref_logps, old_logps, values, rewards, mask):
        adv, returns = self.compute_gae(rewards, values, mask)
        ratio = torch.exp(cur_logps - old_logps)
        kl_ratio = (cur_logps - ref_logps).clamp(min=-10, max=10)
        adv = (adv - adv.mean(dim=-1, keepdim=True)) / (adv.std(dim=-1, keepdim=True) + 1e-8)
        adv = adv - kl_ratio
        loss = torch.min(ratio * adv, torch.clamp(ratio, 1 + self.epsilon, 1 - self.epsilon) * adv)
        loss = -(loss * mask.float()).sum() / mask.sum().clamp(min=1.0)
        return loss, adv, returns


if __name__ == "__main__":
    torch.manual_seed(42)
    B, T, V = 2, 4, 5
    cur_logps = torch.randn(B, T)
    ref_logps = torch.randn(B, T) 
    old_logps = torch.randn(B, T)
    values = torch.randn(B, T)
    rewards = torch.randn(B, T)
    mask = torch.randint(0, 2, (B, T)).float()
    ppo = PPO(beta=0.01, epsilon=0.2, lambda_=0.95, gamma=0.99)
    loss, adv, returns = ppo.get_ppo_loss(cur_logps, ref_logps, old_logps, values, rewards, mask)
    critic_loss = ppo.critic_loss(returns, values, mask)
    print(f"PPO Loss: {loss.item():.4f}, Critic Loss: {critic_loss.item():.4f}")
    
    

## SGD 

In [ ]:
import numpy as np

np.random.seed(42)  # 保证可复现



# ============ SwiGLU 激活函数 ============
def silu(x):
    """Swish/SiLU: x * sigmoid(x)"""
    return x * (1 / (1 + np.exp(-x)))


def silu_backward(x):
    """SiLU 的导数: sigmoid(x) + x * sigmoid(x) * (1 - sigmoid(x))"""
    sig = 1 / (1 + np.exp(-x))
    return sig + x * sig * (1 - sig)



def adam_mlp2(X, y, hidden_dim=64, lr=0.01, epochs=100, batch_size=32,
              beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):
    """
    Adam 优化的 2 层 MLP（MSE 回归）
    
    架构: X -> Linear -> ReLU -> Linear -> y_pred
    Loss: MSE = mean((y_pred - y)^2)
    """
    
    n_samples, n_features = X.shape
    
    # ============ He 初始化（适合 ReLU）============
    W1 = np.random.randn(n_features, hidden_dim) * np.sqrt(2 / n_features)
    b1 = np.zeros(hidden_dim)
    W2 = np.random.randn(hidden_dim, 1) * np.sqrt(2 / hidden_dim)
    b2 = np.zeros(1)
    
    # ============ Adam 一阶矩 m 和二阶矩 v ============
    m_W1, v_W1 = np.zeros_like(W1), np.zeros_like(W1)
    m_b1, v_b1 = np.zeros_like(b1), np.zeros_like(b1)
    m_W2, v_W2 = np.zeros_like(W2), np.zeros_like(W2)
    m_b2, v_b2 = np.zeros_like(b2), np.zeros_like(b2)
    t = 0  # 时间步，用于 bias correction
    
    losses = []
    
    for epoch in range(epochs):
        idx = np.random.permutation(n_samples)
        epoch_loss = 0.0
        n_batches = 0
        
        for i in range(0, n_samples, batch_size):
            t += 1
            Xb = X[idx[i:i+batch_size]]
            yb = y[idx[i:i+batch_size]].reshape(-1, 1)
            
            # ==================== Forward ====================
            # z1 = Xb @ W1 + b1       shape: (batch, hidden)
            # h1 = ReLU(z1)           shape: (batch, hidden)
            # yp = h1 @ W2 + b2       shape: (batch, 1)
            # loss = mean((yp - yb)^2)
            
            z1 = Xb @ W1 + b1
            h1 = np.maximum(0, z1)  # ReLU 
            yp = h1 @ W2 + b2
            
            batch_loss = np.mean((yp - yb) ** 2)
            epoch_loss += batch_loss
            n_batches += 1
            
            # ==================== Backward ====================
            """
            推导过程:
            
            Loss = (1/n) * Σ(yp - y)^2
            
            【Layer 2】
            ∂L/∂yp = (2/n) * (yp - y)              shape: (batch, 1)
            
            ∂L/∂W2 = ∂L/∂yp · ∂yp/∂W2 
                   = h1.T @ ∂L/∂yp                  shape: (hidden, 1)
            
            ∂L/∂b2 = ∂L/∂yp · ∂yp/∂b2 
                   = sum(∂L/∂yp, axis=0)            shape: (1,)
            
            【Layer 1】
            ∂L/∂h1 = ∂L/∂yp · ∂yp/∂h1 
                   = ∂L/∂yp @ W2.T                  shape: (batch, hidden)
            
            ∂L/∂z1 = ∂L/∂h1 · ∂h1/∂z1 
                   = ∂L/∂h1 * ReLU'(z1) 
                   = ∂L/∂h1 * (z1 > 0)              shape: (batch, hidden)
            
            ∂L/∂W1 = ∂L/∂z1 · ∂z1/∂W1 
                   = Xb.T @ ∂L/∂z1                  shape: (features, hidden)
            
            ∂L/∂b1 = sum(∂L/∂z1, axis=0)            shape: (hidden,)
            """
            
            # Layer 2 梯度
            loss_grad = 2 / len(Xb) * (yp - yb)  # ∂L/∂yp
            
            dW2 = h1.T @ loss_grad               # ∂L/∂W2
            db2 = np.sum(loss_grad, axis=0)      # ∂L/∂b2
            
            # Layer 1 梯度
            dz1 = (loss_grad @ W2.T) * (z1 > 0)  # ∂L/∂z1 = ∂L/∂h1 * ReLU'
            dW1 = Xb.T @ dz1                     # ∂L/∂W1
            db1 = np.sum(dz1, axis=0)            # ∂L/∂b1
            
            # ==================== Adam 更新 ====================
            """
            Adam 更新规则:
            m_t = β1 * m_{t-1} + (1-β1) * g_t        一阶矩（动量）
            v_t = β2 * v_{t-1} + (1-β2) * g_t^2      二阶矩（RMSprop）
            m_hat = m_t / (1 - β1^t)                 bias correction
            v_hat = v_t / (1 - β2^t)                 bias correction
            θ = θ - lr * m_hat / (√v_hat + ε)        参数更新
            
            Weight Decay (AdamW 风格):
            θ = θ - lr * (m_hat / (√v_hat + ε) + λ * θ)
            """
            
            for param, grad, m_p, v_p in [
                (W1, dW1, m_W1, v_W1), (b1, db1, m_b1, v_b1),
                (W2, dW2, m_W2, v_W2), (b2, db2, m_b2, v_b2),
            ]:
                # 更新一阶矩和二阶矩
                m_p[:] = beta1 * m_p + (1 - beta1) * grad
                v_p[:] = beta2 * v_p + (1 - beta2) * (grad ** 2)
                
                # Bias correction（修正初始零偏置）
                m_hat = m_p / (1 - beta1 ** t)
                v_hat = v_p / (1 - beta2 ** t)
                
                # Weight decay 只用于权重，不用 bias
                wd = weight_decay if param.ndim == 2 else 0
                
                # 参数更新
                param -= lr * (m_hat / (np.sqrt(v_hat) + eps) + wd * param)
        
        losses.append(epoch_loss / n_batches)
    
    return W1, b1, W2, b2, losses


def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


# ============ 测试代码 ============
X_train = np.random.randn(100, 10)
y_train = np.random.randn(100)
X_test = np.random.randn(20, 10)
y_test = np.random.randn(20)

# 对比 ReLU MLP 和 SwiGLU MLP
print("=" * 50)
print("ReLU MLP:")
W1, b1, W2, b2, losses_relu = adam_mlp2(X_train, y_train, hidden_dim=32, lr=0.01, epochs=50)
z1 = X_test @ W1 + b1
y_pred_relu = (np.maximum(0, z1) @ W2 + b2).ravel()
print(f"Test MSE = {mean_squared_error(y_test, y_pred_relu):.4f}")
print(f"Train loss: {losses_relu[0]:.4f} -> {losses_relu[-1]:.4f}")

# print("\n" + "=" * 50)
# print("SwiGLU MLP:")
# params_swiglu, losses_swiglu = adam_mlp_swiglu(X_train, y_train, hidden_dim=32, lr=0.01, epochs=50)
# z_gate = X_test @ params_swiglu['W_gate'] + params_swiglu['b_gate']
# z_up = X_test @ params_swiglu['W_up'] + params_swiglu['b_up']
# h_test = silu(z_gate) * z_up
# y_pred_swiglu = (h_test @ params_swiglu['W_down'] + params_swiglu['b_down']).ravel()
# print(f"Test MSE = {mean_squared_error(y_test, y_pred_swiglu):.4f}")
# print(f"Train loss: {losses_swiglu[0]:.4f} -> {losses_swiglu[-1]:.4f}")

## Traditional RLs

1. Q-Learning：离线策略，Q(s,a) += lr*(r+γmaxQ'-Q)
2. DQN：经验回放 + 目标网络 + MSE 损失
3. REINFORCE：回合更新，折扣回报归一化，loss=-Σlogπ*R
4. SAC：双 Q + 最大熵 + 重参数化 + 软更新

In [ ]:
import numpy as np
import gymnasium as gym

# 超参
lr = 0.1
gamma = 0.9
epsilon = 0.1
episodes = 500

env = gym.make("CliffWalking-v1")
n_states = env.observation_space.n
n_actions = env.action_space.n

# Q 表
Q = np.zeros((n_states, n_actions))

for ep in range(episodes):
    state, _ = env.reset()
    done = False
    ep_reward = 0
    while not done:
        # epsilon 贪心
        if np.random.uniform() < epsilon:
            action = env.action_space.sample()
        else:
            action = int(np.argmax(Q[state]))

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Q-learning 更新（off-policy: 用 max_a' Q(s', a') bootstrap）
        Q[state, action] = Q[state, action] + lr * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )
        state = next_state
        ep_reward += reward

    if ep % 50 == 0 or ep == episodes - 1:
        print(f"episode={ep:4d}  reward={ep_reward:7.2f}  Q_max={np.abs(Q).max():.3f}")

print("final Q table sample:\n", Q[:5])


In [ ]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque


class DQN(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ReLU(),
            nn.Linear(64, action_dim),
        )

    def forward(self, x):
        return self.fc(x)


# 超参
gamma = 0.99
lr = 1e-3
epsilon = 0.1
buffer_size = 10000
batch_size = 32
target_update_steps = 100   # 每 N 步训练更新一次 target

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

net = DQN(state_dim, action_dim)
target_net = DQN(state_dim, action_dim)
target_net.load_state_dict(net.state_dict())
optimizer = optim.Adam(net.parameters(), lr=lr)
buffer = deque(maxlen=buffer_size)

step_count = 0
for episode in range(500):
    state, _ = env.reset()
    total_reward = 0
    terminated = truncated = False
    while not (terminated or truncated):
        # 选动作
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                action = net(torch.FloatTensor(state)).argmax().item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        buffer.append((state, action, reward, next_state, terminated))   # 用 terminated 做 bootstrap
        total_reward += reward
        state = next_state
        step_count += 1

        # 训练
        if len(buffer) > batch_size:
            batch = random.sample(buffer, batch_size)
            s, a, r, s_, term = zip(*batch)
            s = torch.FloatTensor(np.asarray(s))
            a = torch.LongTensor(a).unsqueeze(1)
            r = torch.FloatTensor(r).unsqueeze(1)
            s_ = torch.FloatTensor(np.asarray(s_))
            term = torch.FloatTensor(term).unsqueeze(1)

            q = net(s).gather(1, a)
            with torch.no_grad():
                max_q = target_net(s_).max(1, keepdim=True)[0]
                target_q = r + gamma * max_q * (1 - term)   # 仅 terminated 归零，truncated 不归零

            loss = nn.MSELoss()(q, target_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # 每 N 步同步 target
            if step_count % target_update_steps == 0:
                target_net.load_state_dict(net.state_dict())

    print(f"Episode {episode}, Total Reward: {total_reward}, Buffer: {len(buffer)}")


In [ ]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim

class Policy(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64), nn.ReLU(),
            nn.Linear(64, action_dim), nn.Softmax(dim=-1)
        )
    def forward(self, x):
        return self.fc(x)

gamma = 0.99
lr = 1e-3

env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

policy = Policy(state_dim, action_dim)
optimizer = optim.Adam(policy.parameters(), lr=lr)

for episode in range(1000):
    state, _ = env.reset()
    log_probs = []
    rewards = []
    
    while True:
        prob = policy(torch.FloatTensor(state))
        action = torch.multinomial(prob, 1).item()
        log_prob = torch.log(prob[action])
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        log_probs.append(log_prob)
        rewards.append(reward)
        state = next_state
        
        if done:
            # 计算折扣回报
            returns = []
            R = 0
            for r in reversed(rewards):
                R = r + gamma * R
                returns.insert(0, R)
            returns = torch.tensor(returns)
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
            
            # 策略梯度损失
            loss = -torch.sum(torch.stack(log_probs) * returns)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            break

In [ ]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal
import random
from collections import deque


# 双 Q 网络
class QNet(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.q1 = nn.Sequential(nn.Linear(s_dim + a_dim, 64), nn.ReLU(), nn.Linear(64, 1))
        self.q2 = nn.Sequential(nn.Linear(s_dim + a_dim, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, s, a):
        x = torch.cat([s, a], dim=-1)
        return self.q1(x), self.q2(x)


# 策略网络
class Actor(nn.Module):
    def __init__(self, s_dim, a_dim, max_a):
        super().__init__()
        self.max_a = max_a
        self.fc = nn.Sequential(nn.Linear(s_dim, 64), nn.ReLU())
        self.mean = nn.Linear(64, a_dim)
        self.log_std = nn.Linear(64, a_dim)

    def forward(self, s):
        x = self.fc(s)
        mean = self.mean(x)
        log_std = torch.clamp(self.log_std(x), -20, 2)
        return mean, log_std

    def sample(self, s):
        mean, log_std = self.forward(s)
        dist = Normal(mean, log_std.exp())
        x = dist.rsample()                                  # reparameterize
        y = torch.tanh(x)
        # tanh 修正项: log(1 - y^2)
        log_prob = dist.log_prob(x) - torch.log(1 - y.pow(2) + 1e-6)
        return y * self.max_a, log_prob.sum(dim=-1, keepdim=True)


# 训练
gamma = 0.99
tau = 0.005
lr = 3e-4
alpha = 0.2                # 熵系数 (SAC 核心项)
batch_size = 32

env = gym.make("Pendulum-v1")
s_dim = env.observation_space.shape[0]
a_dim = env.action_space.shape[0]
max_a = float(env.action_space.high[0])

q_net = QNet(s_dim, a_dim)
q_target = QNet(s_dim, a_dim)
q_target.load_state_dict(q_net.state_dict())
actor = Actor(s_dim, a_dim, max_a)

q_opt = optim.Adam(q_net.parameters(), lr=lr)
a_opt = optim.Adam(actor.parameters(), lr=lr)
buffer = deque(maxlen=10000)

for episode in range(1000):
    s, _ = env.reset()
    terminated = truncated = False
    while not (terminated or truncated):
        # 采样动作
        with torch.no_grad():
            a, _ = actor.sample(torch.FloatTensor(s).unsqueeze(0))
        a = a.squeeze(0).cpu().numpy()
        s_, r, terminated, truncated, _ = env.step(a)
        # 只用 terminated 做 bootstrap，truncated 不应归零
        buffer.append((s, a, r, s_, terminated))
        s = s_

        if len(buffer) > batch_size:
            batch = random.sample(buffer, batch_size)
            s_b, a_b, r_b, s_b_, term_b = zip(*batch)
            s_b = torch.FloatTensor(np.asarray(s_b))
            a_b = torch.FloatTensor(np.asarray(a_b))
            r_b = torch.FloatTensor(r_b).unsqueeze(1)
            s_b_ = torch.FloatTensor(np.asarray(s_b_))
            term_b = torch.FloatTensor(term_b).unsqueeze(1)

            # ---------- 更新 Q ----------
            with torch.no_grad():
                a_, log_p_ = actor.sample(s_b_)
                q1_t, q2_t = q_target(s_b_, a_)
                target_v = torch.min(q1_t, q2_t) - alpha * log_p_
                target_q = r_b + gamma * (1 - term_b) * target_v
            q1, q2 = q_net(s_b, a_b)
            q_loss = nn.MSELoss()(q1, target_q) + nn.MSELoss()(q2, target_q)
            q_opt.zero_grad()
            q_loss.backward()
            q_opt.step()

            # ---------- 更新 Actor (max Q(s,a) + α·H) ----------
            a_pred, log_p = actor.sample(s_b)
            q1_min, q2_min = q_net(s_b, a_pred)
            a_loss = (alpha * log_p - torch.min(q1_min, q2_min)).mean()
            a_opt.zero_grad()
            a_loss.backward()
            a_opt.step()

            # ---------- 软更新 ----------
            for p, tp in zip(q_net.parameters(), q_target.parameters()):
                tp.data.copy_(tau * p.data + (1 - tau) * tp.data)


In [ ]:
import torch
import torch.nn as nn


class fn(nn.Module):
    """SwiGLU FFN（单 expert）"""
    def __init__(self, input_dim: int, output_dim: int, dropout: float = 0.1):
        super().__init__()
        self.up_proj = nn.Linear(input_dim, output_dim * 4)
        self.gated_proj = nn.Linear(input_dim, output_dim * 4)
        self.down_proj = nn.Linear(output_dim * 4, output_dim)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.down_proj(self.dropout(self.act(self.gated_proj(x)) * self.up_proj(x)))


class MoEFN(nn.Module):
    """
    Top-k MoE，带 Switch-Transformer 风格的 aux load-balance loss:
        aux = alpha * N * Σ_i (f_i · P_i)
        f_i = 路由到 expert i 的 token 比例 (实际计数)
        P_i = 路由到 expert i 的平均概率
    """
    def __init__(self, input_dim, output_dim, num_experts=4, top_k=2, dropout=0.1, aux_coef=0.01):
        super().__init__()
        self.experts = nn.ModuleList([fn(input_dim, output_dim, dropout) for _ in range(num_experts)])
        self.router = nn.Linear(input_dim, num_experts)
        self.top_k = top_k
        self.num_experts = num_experts
        self.aux_coef = aux_coef

    def forward(self, x):
        bsz, seq_len, _ = x.size()
        N = bsz * seq_len

        router_logits = self.router(x)                          # [B, T, E]
        router_probs = torch.softmax(router_logits, dim=-1)     # [B, T, E]

        # Top-k 路由
        topk_probs, topk_idx = torch.topk(router_probs, self.top_k, dim=-1)   # [B, T, k]
        # 归一化 top-k 概率，使加权后总和为 1（标准 MoE 做法）
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True).clamp_min(1e-6)

        # 计算所有 expert 输出
        expert_outputs = torch.stack([expert(x) for expert in self.experts], dim=2)   # [B, T, E, D]
        # 按路由索引 gather: [B, T, k, D]
        idx = topk_idx.unsqueeze(-1).expand(-1, -1, -1, expert_outputs.size(-1))
        topk_outputs = torch.gather(expert_outputs, dim=2, index=idx)
        # 加权聚合
        output = (topk_outputs * topk_probs.unsqueeze(-1)).sum(dim=2)               # [B, T, D]

        # ---------- 负载均衡 aux loss (Switch Transformer) ----------
        # f_i: 每个 expert 实际接收的 token 比例（按 argmax 计）
        with torch.no_grad():
            argmax_idx = router_logits.argmax(dim=-1)                    # [B, T]
            one_hot = torch.nn.functional.one_hot(argmax_idx, self.num_experts).float()  # [B, T, E]
            f = one_hot.flatten(0, 1).mean(dim=0)                        # [E]
        P = router_probs.flatten(0, 1).mean(dim=0)                       # [E]
        aux_loss = self.aux_coef * self.num_experts * (f * P).sum()

        return output, aux_loss


moe = MoEFN(input_dim=16, output_dim=32, num_experts=4, top_k=2)
x = torch.randn(8, 10, 16)
output, aux_loss = moe(x)
print(f"Output shape: {output.shape}, Aux loss: {aux_loss.item():.4f}")


In [ ]:
# Top-p

import torch


def get_topp(logits, p, temperature=1.0):
    logits = logits / temperature
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)
    mask = cumulative_probs > p
    # mask[..., 1:] = mask[..., :-1].clone()
    mask = torch.roll(mask, shifts=1, dims=-1)
    mask[..., 0] = False
    sorted_logits[mask] = float('-inf')
    return sorted_logits.scatter(1, sorted_indices, sorted_logits)
    

def get_topk(logits, k):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    topk_logits = sorted_logits[:, :k]
    topk_indices = sorted_indices[:, :k]
    return topk_logits, topk_indices


def beam_search(logits, beam_width):
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    topk_logits = sorted_logits[:, :beam_width]
    topk_indices = sorted_indices[:, :beam_width]
    return topk_logits, topk_indices



logits = torch.tensor([[0.1, 0.2, 0.3, 0.4]])
p = 0.6
filtered_logits = get_topp(logits, p)
print(filtered_logits)

In [ ]:
import numpy as np

x = np.random.rand(1000000)
y = np.random.rand(1000000)

val = (x**2 + y**2) <= 1


print(4 * val.sum() / len(x))


In [ ]:
import torch

a = torch.randn(2, 4, 8) # bsz, seq_len, hidden_size

def safe_softmax(logits):
    max_val = torch.max(logits, dim=-1, keepdim=True).values
    logits = torch.exp(logits - max_val)
    return logits / torch.sum(logits, dim=-1, keepdim=True)


def ce_loss(logits, targets):
    delta = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    print(targets.unsqueeze(-1).shape, delta.shape)
    return -torch.mean(torch.gather(delta, dim=-1, index=targets.unsqueeze(-1)).squeeze(-1))


logits = torch.randn(2, 4, 8)
targets = torch.randint(0, 8, (2, 4))
print(ce_loss(logits, targets))
# probs = safe_softmax(logits)
# print(probs)
    



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict


def get_mean(points, k=4, max_iter=1000):
    rng = np.random.default_rng(42)
    n = len(points)
    center_points = points[rng.integers(0, n, size=k)].copy()
    for _ in range(max_iter):
        # E 步：分配到最近中心
        dists = np.linalg.norm(points[:, None, :] - center_points[None, :, :], axis=-1)  # [n, k]
        cluster_idxs = dists.argmin(axis=-1)                                            # [n]
        # M 步：更新中心（空簇保护：保持原中心）
        new_centers = center_points.copy()
        for i in range(k):
            mask = cluster_idxs == i
            if mask.any():
                new_centers[i] = points[mask].mean(axis=0)
        # 收敛判定
        if np.allclose(new_centers, center_points):
            center_points = new_centers
            break
        center_points = new_centers
    # 最终聚类索引
    dists = np.linalg.norm(points[:, None, :] - center_points[None, :, :], axis=-1)
    return dists.argmin(axis=-1)


rand_points = np.random.randn(100, 2)
clusters = get_mean(rand_points, k=4)
plt.scatter(rand_points[:, 0], rand_points[:, 1], c=clusters)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("K-means")
plt.show()


## LLM Blocks

In [ ]:
## Decoder layer

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, List


# ==================== RoPE ====================
# 注：LLaMA 风格 rotate_half 实现，cos/sin 需要 head_dim 维（复制两份）
def precompute_freqs_cis(dim: int, end: int = int(32 * 1024), rope_base: float = 1e6,
                         rope_scaling: Optional[dict] = None, device: torch.device = None):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2, device=device)[: (dim // 2)].float() / dim))
    if rope_scaling is not None:
        orig_max = rope_scaling.get("original_max_position_embeddings", 2048)
        factor = rope_scaling.get("factor", 4)
        beta_fast = rope_scaling.get("beta_fast", 4.0)
        beta_slow = rope_scaling.get("beta_slow", 1.0)
        if end / orig_max > 1.0:
            corr_dim = next((i for i in range(dim // 2) if 2 * math.pi / freqs[i] > orig_max), dim // 2)
            power = torch.arange(dim // 2, device=freqs.device).float() / max(dim // 2 - 1, 1)
            beta = beta_slow + (beta_fast - beta_slow) * power
            scale = torch.where(
                torch.arange(dim // 2, device=freqs.device) < corr_dim,
                (beta * factor - beta + 1) / (beta * factor),
                1.0 / factor,
            )
            freqs = freqs * scale
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()                     # [end, dim//2]
    # 复制成完整 head_dim 维（配合 rotate_half）
    cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)   # [end, dim]
    sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return cos, sin


def apply_rotary_pos_emb(q, k, cos, sin):
    """
    q, k: [bsz, num_heads, seq_len, head_dim]
    cos, sin: [seq_len, head_dim]
    """
    def rotate_half(x):
        x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
        return torch.cat((-x2, x1), dim=-1)
    cos = cos.unsqueeze(0).unsqueeze(0)   # [1, 1, seq_len, head_dim]
    sin = sin.unsqueeze(0).unsqueeze(0)
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


# ==================== RMSNorm ====================
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight


# ==================== Attention ====================
class Attention(nn.Module):
    def __init__(self, hidden_dim, num_q_head, num_kv_head, dropout=0.0):
        super().__init__()
        self.num_q_head = num_q_head
        self.num_kv_head = num_kv_head
        self.head_dim = hidden_dim // num_q_head
        self.q_proj = nn.Linear(hidden_dim, num_q_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, num_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, num_kv_head * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_q_head * self.head_dim, hidden_dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None, kv_cache: Optional[List[torch.Tensor]] = None):
        bsz, in_seq_len, _ = x.shape
        q = self.q_proj(x).reshape(bsz, in_seq_len, self.num_q_head, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).reshape(bsz, in_seq_len, self.num_kv_head, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).reshape(bsz, in_seq_len, self.num_kv_head, self.head_dim).transpose(1, 2)

        start_pos = kv_cache[0].size(2) if kv_cache is not None else 0

        cos_all, sin_all = precompute_freqs_cis(self.head_dim, start_pos + in_seq_len, device=x.device)
        cos = cos_all[start_pos: start_pos + in_seq_len]
        sin = sin_all[start_pos: start_pos + in_seq_len]
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        if kv_cache is not None:
            kv_cache[0] = torch.cat([kv_cache[0], k], dim=2)
            kv_cache[1] = torch.cat([kv_cache[1], v], dim=2)
            k, v = kv_cache
        kv_len = k.size(2)

        n_rep = self.num_q_head // self.num_kv_head
        if n_rep > 1:
            k = k.repeat_interleave(n_rep, dim=1)
            v = v.repeat_interleave(n_rep, dim=1)

        score = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)

        if attention_mask is not None:
            score = score.masked_fill(attention_mask[:, None, None, :] == 0, float("-inf"))

        if in_seq_len > 1:
            causal_mask = torch.triu(
                torch.full((in_seq_len, kv_len), float("-inf"), device=x.device),
                diagonal=kv_len - in_seq_len + 1,
            )
            score = score + causal_mask

        attn_weights = torch.softmax(score, dim=-1)
        attn_weights = self.dropout(attn_weights)
        out = (attn_weights @ v).transpose(1, 2).reshape(bsz, in_seq_len, -1)
        return self.o_proj(out), kv_cache


# ==================== SwiGLU FFN ====================
class SwiGLU(nn.Module):
    def __init__(self, hidden_dim, intermediate_dim=None):
        super().__init__()
        if intermediate_dim is None:
            intermediate_dim = ((int(hidden_dim * 8 / 3) + 255) // 256) * 256
        self.gate_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.up_proj = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.down_proj = nn.Linear(intermediate_dim, hidden_dim, bias=False)

    def silu(self, x):
        return x * torch.sigmoid(x)

    def forward(self, x):
        return self.down_proj(self.silu(self.gate_proj(x)) * self.up_proj(x))


# ==================== Decoder ====================
class DecoderLayer(nn.Module):
    def __init__(self, hidden_dim, num_q_head, num_kv_head, intermediate_dim=None):
        super().__init__()
        self.attn_norm = RMSNorm(hidden_dim)
        self.attn = Attention(hidden_dim, num_q_head, num_kv_head)
        self.ffn_norm = RMSNorm(hidden_dim)
        self.ffn = SwiGLU(hidden_dim, intermediate_dim)

    def forward(self, x, attention_mask=None, kv_cache=None):
        h = self.attn_norm(x)
        attn_out, kv_cache = self.attn(h, attention_mask, kv_cache)
        x = x + attn_out
        h = self.ffn_norm(x)
        x = x + self.ffn(h)
        return x, kv_cache


class Decoder(nn.Module):
    def __init__(self, vocab_size, hidden_dim, num_layers, num_q_head, num_kv_head,
                 intermediate_dim=None, max_seq_len=4096):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.num_kv_head = num_kv_head
        self.head_dim = hidden_dim // num_q_head
        self.embed_tokens = nn.Embedding(vocab_size, hidden_dim)
        self.layers = nn.ModuleList([
            DecoderLayer(hidden_dim, num_q_head, num_kv_head, intermediate_dim)
            for _ in range(num_layers)
        ])
        self.norm = RMSNorm(hidden_dim)
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)
        self.lm_head.weight = self.embed_tokens.weight  # weight tying

    def init_kv_cache(self, bsz, device=None):
        return [
            [
                torch.zeros(bsz, self.num_kv_head, 0, self.head_dim, device=device),
                torch.zeros(bsz, self.num_kv_head, 0, self.head_dim, device=device),
            ]
            for _ in range(self.num_layers)
        ]

    def forward(self, input_ids, attention_mask=None, kv_caches=None):
        x = self.embed_tokens(input_ids)
        if kv_caches is None:
            kv_caches = [None] * self.num_layers
        new_caches = []
        for i, layer in enumerate(self.layers):
            x, cache = layer(x, attention_mask, kv_caches[i])
            new_caches.append(cache)
        x = self.norm(x)
        return self.lm_head(x), new_caches


# ==================== 测试 ====================
H, L, Q, KV, V = 2048, 16, 32, 8, 32000
model = Decoder(V, H, L, Q, KV)

ids = torch.randint(0, V, (2, 64))
logits, _ = model(ids)
print(f"prefill logits: {logits.shape}")

# Decode + KV cache
kv = model.init_kv_cache(2)
_, kv = model(ids[:, :32], kv_caches=kv)
_, kv = model(ids[:, 32:33], kv_caches=kv)
print(f"kv cache len after decode: {kv[0][0].size(2)}")  # 33


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from typing import List, Dict

class OnPolicyDistillation:
    def __init__(
        self,
        student_model: nn.Module,          # 学生模型 (当前策略 π_θ)
        teacher_models: List[nn.Module],   # 教师模型列表 (π_Ei)
        teacher_weights: List[float],      # 每个教师对应的权重 w_i
        temperature: float = 1.0,
    ):
        self.student = student_model
        self.teachers = teacher_models
        self.weights = teacher_weights
        self.temperature = temperature
        
        # 确保权重数量与教师数量一致
        assert len(self.teachers) == len(self.weights)

    def _get_logits(
        self, 
        model: nn.Module, 
        input_ids: torch.Tensor, 
        attention_mask: torch.Tensor
    ) -> torch.Tensor:
        """获取模型在当前输入序列上的全词表 logits（带温度调节）"""
        with torch.no_grad() if model is not self.student else torch.enable_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            logits = outputs.logits / self.temperature   # [batch, seq_len, vocab_size]
        return logits

    def compute_opd_loss(
        self,
        on_policy_ids: torch.Tensor,      # 学生模型生成的 on-policy 序列 [B, L]
        attention_mask: torch.Tensor,     # [B, L]
    ) -> torch.Tensor:
        """
        根据已经采样好的 on-policy 轨迹，计算 OPD 损失：
        L = Σ_i w_i * D_KL(π_θ || π_Ei)
        """
        total_loss = 0.0
        
        # 学生 logits（需要梯度）
        student_logits = self._get_logits(self.student, on_policy_ids, attention_mask)
        student_log_probs = F.log_softmax(student_logits, dim=-1)
        student_probs = F.softmax(student_logits, dim=-1)
        
        # 逐教师计算反向 KL 散度
        for teacher, weight in zip(self.teachers, self.weights):
            teacher_logits = self._get_logits(teacher, on_policy_ids, attention_mask)
            teacher_log_probs = F.log_softmax(teacher_logits, dim=-1)
            
            # D_KL( student || teacher ) = Σ_x p_student(x) * (log p_student(x) - log p_teacher(x))
            # 在序列维度求和后求平均
            kl_per_token = torch.sum(
                student_probs * (student_log_probs - teacher_log_probs), dim=-1
            )  # [B, L]
            # 只对有效 token（非 padding）部分计算损失
            masked_kl = kl_per_token * attention_mask
            loss = masked_kl.sum() / attention_mask.sum()
            
            total_loss += weight * loss
            
        return total_loss

    def training_step(
        self,
        prompts: torch.Tensor,           # 输入提示 [B, L_prompt]
        max_new_tokens: int,
        optimizer: torch.optim.Optimizer,
    ) -> float:
        """单步 OPD 训练：采样 -> 计算损失 -> 更新"""
        self.student.eval()  # 采样时学生模型处于 eval 模式
        self.teachers = [t.eval() for t in self.teachers]  # 教师永远 eval
        
        # 1. 从学生策略中采样获得 on-policy 序列
        with torch.no_grad():
            generated = self.student.generate(
                prompts,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=self.temperature,
                pad_token_id=0
            )
        
        # 构建 attention mask（假设 padding 在左侧或右侧，简单处理为全1）
        attention_mask = (generated != 0).float()
        
        self.student.train()  # 学生模型切回训练模式
        
        # 2. 计算 OPD 损失（学生部分需要梯度）
        loss = self.compute_opd_loss(generated, attention_mask)
        
        # 3. 反向传播与优化器更新
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        return loss.item()


# ----------------- 使用示例 -----------------
# 假设已经定义了 student_model, teacher_list, weights
# opd_trainer = OnPolicyDistillation(student_model, teacher_list, weights, temperature=1.0)
# optimizer = torch.optim.AdamW(student_model.parameters(), lr=1e-5)
#
# for batch_prompts in dataloader:
#     loss = opd_trainer.training_step(batch_prompts, max_new_tokens=128, optimizer=optimizer)
#     print(f"OPD loss: {loss}")

## Others

In [ ]:
import torch
from vllm import LLM, SamplingParams


prompts = [
    "请用中文介绍一下你自己。",
    "什么是机器学习？"
]

model = LLM(model="Qwen/Qwen2.5-7B-Instruct", tensor_parallel_size=1, max_model_len=4096, gpu_memory_utilization=0.9)

outputs = model.generate(prompts, SamplingParams(temperature=0.7, top_p=0.9, max_tokens=512))

for o in outputs:
    print(o.outputs[0].text)

In [ ]:
import torch

# scatter_add_(dim, index, src): src 与 index 必须同形状
src = torch.ones((2, 4))
index = torch.tensor([[0, 1, 2, 0],
                      [1, 2, 0, 0]])
out = torch.zeros(3, 4, dtype=src.dtype)
out.scatter_add_(0, index, src)   # 按 dim=0 把 src 累加到 out[index[i,j], j]
print(out)
# 期望:
# out[0,j] = src[0,j](index=0) + src[1,j](index=0 if j in {2,3})
# 第 0 列: out[0,0]=1 (来自 row0), out[1,0]=1 (来自 row1)
# 第 2 列: out[0,2]=1 (row1 index=0), out[2,2]=1 (row0 index=2)


In [ ]:
"""
Qwen2.5-0.5B SFT 最小可跑 demo
- prompt 部分标签置 -100,只对 answer 计 loss
- 手动 CE 展示原理(也可以直接传 labels 给 model,等价)
"""

import torch
import torch.nn.functional as F
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "/Users/liushz/Codes/others/models/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
# Qwen 默认没设 pad_token,用 eos 兜底
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 用 fp32 跑(CPU 演示);真 GPU 上换成 torch.bfloat16
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

# ============ 数据 ============
qa_datas = [
    {"question": "请用中文介绍一下你自己。",
     "answer": "我是一个人工智能助手,能够回答各种问题,提供信息和帮助。"},
    {"question": "什么是机器学习?",
     "answer": "机器学习是一种人工智能技术,通过算法和统计模型使计算机系统能够从数据中学习和改进性能,而无需明确编程。"},
    {"question": "请解释一下深度学习的基本概念。",
     "answer": "深度学习是机器学习的一个分支,使用多层神经网络来建模复杂的数据模式和特征表示。"},
    {"question": "什么是自然语言处理(NLP)?",
     "answer": "自然语言处理是人工智能的一个领域,涉及计算机与人类语言的交互,包括理解、生成和分析自然语言。"},
]
qa_datas = qa_datas * 100  # 扩展数据集以便训练

# ============ Chat Template(用 Qwen2.5 官方格式)============
PROMPT_TEMPLATE = "<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n"
ANSWER_SUFFIX = "<|im_end|>"


def build_example(q: str, a: str, max_len: int):
    """把一条 QA 编码成 (input_ids, attention_mask, labels),prompt 部分标签 = -100"""
    prompt_text = PROMPT_TEMPLATE.format(q=q)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    answer_ids = tokenizer(a + ANSWER_SUFFIX, add_special_tokens=False).input_ids

    full_ids = prompt_ids + answer_ids
    full_ids = full_ids[:max_len]                            # 截断
    valid_len = len(full_ids)
    pad_len = max_len - valid_len
    input_ids = torch.tensor(
        full_ids + [tokenizer.pad_token_id] * pad_len, dtype=torch.long)
    attention_mask = torch.tensor(
        [1] * valid_len + [0] * pad_len, dtype=torch.long)

    # label: prompt 段 + pad 段都置 -100,只有 answer 段参与 loss
    labels = input_ids.clone()
    labels[:len(prompt_ids)] = -100
    labels[attention_mask == 0] = -100
    return input_ids, attention_mask, labels


# ============ 训练超参 ============
B, T = 2, 64          # T=64 容得下最长样本;之前 32 偏短会截断 answer
epoch = 2
lr = 1e-5
optimizer = optim.AdamW(model.parameters(), lr=lr)
model.train()

# ============ 训练循环 ============
for e in range(epoch):
    total_loss, total_tokens, n_batch = 0.0, 0, 0

    for i in range(0, len(qa_datas), B):
        batch = qa_datas[i:i + B]

        # 拼成 [B, T]
        input_ids, attn, labels = zip(*[build_example(d["question"], d["answer"], T) for d in batch])
        input_ids = torch.stack(input_ids)
        attn = torch.stack(attn)
        labels = torch.stack(labels)

        # 前向
        outputs = model(input_ids=input_ids, attention_mask=attn)
        logits = outputs.logits                            # [B, T, V]

        # 手动 CE: shift 后用 ignore_index=-100
        # 也可以直接 outputs = model(..., labels=labels),HF 内部逻辑等价
        shift_logits = logits[:, :-1, :].contiguous()      # 预测位置 t+1
        shift_labels = labels[:, 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
        )

        # 反向
        optimizer.zero_grad()
        loss.backward()
        # 可选:grad_clip
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * (shift_labels != -100).sum().item()
        total_tokens += (shift_labels != -100).sum().item()
        n_batch += 1

        if n_batch % 50 == 0:
            print(f"[epoch {e+1}/{epoch}] step {n_batch:4d}  "
                  f"loss={loss.item():.4f}  ppl={torch.exp(loss).item():.2f}")

    avg_loss = total_loss / max(total_tokens, 1)
    print(f"=== epoch {e+1} done, avg_token_loss={avg_loss:.4f}, ppl={torch.exp(torch.tensor(avg_loss)).item():.2f} ===")


# ============ 训练后做一次推理验证 ============
print("\n=== 采样生成 ===")
model.eval()
prompt_text = PROMPT_TEMPLATE.format(q="什么是机器学习?")
inputs = tokenizer(prompt_text, return_tensors="pt")
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
print(tokenizer.decode(out[0][inputs.input_ids.size(1):], skip_special_tokens=True))


In [151]:
a = [(1,2,3) for _ in range(10)]
print(*a)
print(list(zip(*a)))

(1, 2, 3) (1, 2, 3) (1, 2, 3) (1, 2, 3) (1, 2, 3) (1, 2, 3) (1, 2, 3) (1, 2, 3) (1, 2, 3) (1, 2, 3)
[(1, 1, 1, 1, 1, 1, 1, 1, 1, 1), (2, 2, 2, 2, 2, 2, 2, 2, 2, 2), (3, 3, 3, 3, 3, 3, 3, 3, 3, 3)]


In [ ]:
import torch


def compute_length_penalty(seq_lengths, max_length, cache_length):
    """
    Soft Overlong Punishment (DAPO 核心特性 4):
        R_length(L) = 0,                                   if L <= L_max - L_cache
                     = ((L_max - L_cache) - L) / L_cache,  if L_max - L_cache < L <= L_max
                     = -1,                                 if L > L_max
    """
    safe_boundary = max_length - cache_length
    length_penalty = torch.zeros_like(seq_lengths, dtype=torch.float32)

    warning_zone_mask = (seq_lengths > safe_boundary) & (seq_lengths <= max_length)
    length_penalty[warning_zone_mask] = (safe_boundary - seq_lengths[warning_zone_mask]) / cache_length

    critical_zone_mask = seq_lengths > max_length
    length_penalty[critical_zone_mask] = -1.0
    return length_penalty


def apply_overlong_filter(seq_lengths, max_length):
    """Overlong Filtering: 严格超长 (L > L_max) 的样本在 loss 中过滤掉"""
    return (seq_lengths.unsqueeze(-1) <= max_length).float()   # [bsz, 1]


def dynamic_sampling_mask(rewards, group_size):
    """
    Dynamic Sampling / Group Filtering: 组内奖励无方差则跳过整组
    返回有效样本索引列表
    """
    batch_size = rewards.shape[0]
    num_groups = batch_size // group_size
    valid_indices = []
    for g in range(num_groups):
        s, e = g * group_size, (g + 1) * group_size
        if rewards[s:e].std() > 0:
            valid_indices.extend(range(s, e))
    return valid_indices


def dapo_loss(new_logps, old_logps, ref_logps, attention_mask, rewards, seq_lengths,
              epsilon_low=0.2, epsilon_high=0.28, kl_coeff=0.0,
              max_length=2048, cache_length=256,
              group_size=16, apply_length_penalty=True, apply_overlong_filtering=True):
    """DAPO 五大特性: Clip-Higher + 动态采样 + 超长过滤 + 软超长惩罚 + Token-level Loss"""
    valid_indices = dynamic_sampling_mask(rewards, group_size)
    if not valid_indices:
        return torch.tensor(0.0, device=new_logps.device, requires_grad=True)
    idx = torch.tensor(valid_indices, device=new_logps.device)
    new_logps = new_logps[idx]
    old_logps = old_logps[idx]
    ref_logps = ref_logps[idx]
    attention_mask = attention_mask[idx]
    rewards = rewards[idx]
    seq_lengths = seq_lengths[idx]

    if apply_length_penalty:
        rewards = rewards + compute_length_penalty(seq_lengths, max_length, cache_length)

    # 组内相对优势
    token_rewards = rewards.unsqueeze(-1).repeat(1, new_logps.shape[1])
    token_adv = (token_rewards - rewards.mean()) / (rewards.std() + 1e-8)

    ratio = torch.exp(new_logps - old_logps)
    # Clip-Higher
    pg1 = -token_adv * ratio
    pg2 = -token_adv * torch.clamp(ratio, 1 - epsilon_low, 1 + epsilon_high)
    pg_losses = torch.maximum(pg1, pg2)

    # k3 KL estimator
    kl_ratio = torch.exp(new_logps - ref_logps)
    k3_kl = kl_coeff * (kl_ratio - 1 - torch.log(kl_ratio))

    token_level_loss = pg_losses + k3_kl

    if apply_overlong_filtering:
        attention_mask = attention_mask * apply_overlong_filter(seq_lengths, max_length)

    total_valid_tokens = attention_mask.sum()
    if total_valid_tokens == 0:
        return torch.tensor(0.0, device=new_logps.device, requires_grad=True)
    return (token_level_loss * attention_mask).sum() / total_valid_tokens


# ============ 演示 ============
bsz, seq_len = 8, 32
group_size = 4
max_length = 24    # 注意: 远小于 seq_len，制造超长样本
cache_length = 4   # safe_boundary = 20

new_logps = torch.randn((bsz, seq_len))
old_logps = torch.randn((bsz, seq_len))
ref_logps = torch.randn((bsz, seq_len))
attention_mask = torch.ones((bsz, seq_len))

# 测试 8 条样本（2 组）
seq_lengths = torch.tensor([18, 22, 26, 28,   # 第 1 组：18 正常, 22 警告区, 26/28 严重超长
                            20, 21, 23, 24])  # 第 2 组：边缘
# 真实 mask：超出实际长度的部分置 0（模拟 padding）
for i, l in enumerate(seq_lengths):
    attention_mask[i, min(l, seq_len):] = 0

# 第 1 组奖励有方差 -> 有效；第 2 组完全相同 -> 被过滤
rewards = torch.tensor([0.9, 0.1, 0.8, 0.2,
                        0.5, 0.5, 0.5, 0.5])

loss = dapo_loss(
    new_logps, old_logps, ref_logps, attention_mask, rewards, seq_lengths,
    epsilon_low=0.2, epsilon_high=0.28,
    kl_coeff=0.0,
    max_length=max_length, cache_length=cache_length,
    group_size=group_size,
    apply_length_penalty=True, apply_overlong_filtering=True,
)
print(f"DAPO 损失: {loss.item():.4f}")
print("长度惩罚示例:", compute_length_penalty(torch.tensor([18., 22., 26.]), 24, 4).tolist())
# 期望: [0.0, -0.5, -1.0]  (18<=20 -> 0;  20<22<=24 -> (20-22)/4=-0.5;  26>24 -> -1)


In [ ]:
# 定义原函数
def f(x):
    return (x**3 - 8)**2

# 一阶导数
def df(x):
    return 2 * (x**3 - 8) * (3*x**2)

# 二阶导数
def ddf(x):
    return 2 * ((3*x**2)**2 + (x**3 - 8) * 6*x)
# 梯度下降
def gradient_descent(x0, lr=0.0001, eps=1e-5, max_iter=10000):
    x = x0
    for i in range(max_iter):
        grad = df(x)
        # 收敛条件：导数接近0
        if abs(x**3 - 8) < eps:
            break
        x = x - lr * grad
    return x, f(x)

# 运行
x_min, y_min = gradient_descent(x0=4.0)
print("===== 梯度下降结果 =====")
print(f"极小值点 x = {x_min:.6f}")
print(f"极小值   y = {y_min:.6f}")


# 牛顿法
def newton_method(x0, eps=1e-6, max_iter=100):
    x = x0
    for i in range(max_iter):
        grad = df(x)
        hess = ddf(x)
        if abs(x**3 - 8) < eps:
            break
        # 迭代公式 x = x - f'(x)/f''(x)
        x = x - grad / hess
    return x, f(x)

# 运行
x_min_n, y_min_n = newton_method(x0=4.0)
print("\n===== 牛顿法结果 =====")
print(f"极小值点 x = {x_min_n:.6f}")
print(f"极小值   y = {y_min_n:.6f}")


In [ ]:
import numpy as np  

# 稳定版 softmax（减最大值防溢出）
def softmax(z):
    # z: 1D array
    z_max = np.max(z)
    exp_z = np.exp(z - z_max)
    return exp_z / np.sum(exp_z)

# 手撕 Softmax 雅可比矩阵 (Jacobian)
def softmax_jacobian(z):
    y = softmax(z)
    n = len(y)
    # 初始化雅可比矩阵 [n, n]
    jac = np.zeros((n, n))
    
    for i in range(n):
        for j in range(n):
            if i == j:
                jac[i][j] = y[i] * (1 - y[i])
            else:
                jac[i][j] = -y[i] * y[j]
    return jac

# 测试
z = np.array([1.0, 2.0, 3.0])
y = softmax(z)
jac = softmax_jacobian(z)

print("Softmax 输出 y:\n", y)
print("\nSoftmax 雅可比矩阵 ∂y_i/∂z_j:\n", jac)


In [ ]:
import torch
import torch.nn.functional as F


def get_labels(input_ids, prompt_len, pad_token_id=0):
    """
    构建标签张量，非填充位置为输入ID，填充位置为 -100（CrossEntropyLoss 的 ignore_index）
    
    参数:
        input_ids (torch.Tensor): 输入的 token ID 张量，形状 (batch_size, seq_len)
        prompt_len (torch.Tensor): prompt 的长度，前 prompt_len 个 token 不计算损失
        pad_token_id (int): 填充 token 的 ID，默认值为 0
    
    返回:
        torch.Tensor: 标签张量，形状与 input_ids 相同，填充位置为 -100
    """
    positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0) # [1, seq_len]
    prompt_len = prompt_len.unsqueeze(1)  # [batch_size, 1]
    labels = input_ids.clone()
    labels[positions < prompt_len] = -100
    labels[labels == pad_token_id] = -100
    return labels




import torch.nn as nn
import torch.optim as optim


model = nn.Linear(32, 8)
inputs = torch.randn(128, 32)
targets = torch.randint(0, 8, (128,))

optimizer = optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def grad_clip(model, max_norm):
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += torch.norm(p.grad).item() ** 2
    total_norm = total_norm ** 0.5
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1:
        for p in model.parameters():
            if p.grad is not None:
                p.grad.data.mul_(clip_coef)


for epoch in range(10):
    # model.train()
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    grad_clip(model, max_norm=1.0)
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")
    


In [ ]:
"""
分类指标手撕:Precision / Recall / F1 / AUC

约定:
    pred   : 模型预测 (0/1 二值,或概率)
    target : 真实标签 (0/1)
"""

import torch


# ==================== 1. Precision / Recall / F1 ====================
# 修复点:之前的版本 P 和 R 写反了
#   precision = TP / (TP + FP) = TP / sum(pred==1)
#   recall    = TP / (TP + FN) = TP / sum(target==1)

def precision_recall_f1(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-8):
    """pred / target 都是 0/1 整型"""
    tp = ((pred == 1) & (target == 1)).sum().float()
    fp = ((pred == 1) & (target == 0)).sum().float()
    fn = ((pred == 0) & (target == 1)).sum().float()

    precision = tp / (tp + fp).clamp_min(eps)        # 预测为 1 里有多少真的是 1
    recall    = tp / (tp + fn).clamp_min(eps)        # 真实为 1 里有几个被找回来了
    f1 = 2 * precision * recall / (precision + recall).clamp_min(eps)
    return precision.item(), recall.item(), f1.item()


# ==================== 2. AUC (Mann-Whitney U) ====================
# 概率解释:AUC = P(score(正样本) > score(负样本))
# 公式:对所有 (i 正, j 负) 对,i 的分数比 j 高(平局算 0.5)的比例

def auc_mann_whitney(scores: torch.Tensor, target: torch.Tensor) -> float:
    """
    scores: [N] 模型输出的概率/分数
    target: [N] 0/1 标签
    """
    pos_scores = scores[target == 1]                 # [N_pos]
    neg_scores = scores[target == 0]                 # [N_neg]
    if pos_scores.numel() == 0 or neg_scores.numel() == 0:
        return float("nan")

    # 两两比较: pos_score[i] vs neg_score[j]
    # wins[i, j] = 1 if pos > neg, 0.5 if tie, 0 if pos < neg
    diff = pos_scores.unsqueeze(1) - neg_scores.unsqueeze(0)   # [N_pos, N_neg]
    wins = (diff > 0).float() + (diff == 0).float() * 0.5
    return wins.mean().item()


# ==================== 3. AUC (阈值扫描,等价于 ROC 曲线下面积) ====================

def auc_threshold_sweep(scores: torch.Tensor, target: torch.Tensor) -> float:
    """用所有不同 score 作为阈值,画 ROC 曲线,梯形积分"""
    # 按 score 降序排
    order = scores.argsort(descending=True)
    scores_sorted = scores[order]
    target_sorted = target[order].float()

    P = target_sorted.sum().item()
    N = len(target_sorted) - P
    if P == 0 or N == 0:
        return float("nan")

    # 每个位置累计 tp / fp
    cum_tp = target_sorted.cumsum(0)
    cum_fp = (1 - target_sorted).cumsum(0)
    tpr = cum_tp / P                       # 真阳率
    fpr = cum_fp / N                       # 假阳率

    # 把 (0,0) 和 (1,1) 端点补全
    tpr = torch.cat([torch.tensor([0.0]), tpr])
    fpr = torch.cat([torch.tensor([0.0]), fpr])

    # 梯形法积分
    return float(torch.trapz(tpr, fpr))


# ==================== 验证 ====================
torch.manual_seed(0)

# 二分类预测
target = torch.randint(0, 2, (10,))
pred = (torch.rand(10) > 0.5).long()                  # 0/1 预测
scores = torch.rand(10)                                # 连续概率分数

p, r, f1 = precision_recall_f1(pred, target)
print(f"Precision = {p:.4f}   Recall = {r:.4f}   F1 = {f1:.4f}")

auc1 = auc_mann_whitney(scores, target)
auc2 = auc_threshold_sweep(scores, target)
print(f"AUC (Mann-Whitney)  = {auc1:.4f}")
print(f"AUC (threshold sweep)= {auc2:.4f}")
assert abs(auc1 - auc2) < 1e-6, "两种实现应该一致"
print("✅ 两种 AUC 实现数值一致")

# 对照 sklearn(若装了)
try:
    from sklearn.metrics import roc_auc_score
    ref = roc_auc_score(target.numpy(), scores.numpy())
    print(f"AUC (sklearn ref)   = {ref:.4f}")
    assert abs(auc1 - ref) < 1e-6
    print("✅ 与 sklearn 一致")
except ImportError:
    print("(sklearn 未装,跳过对照)")

# 极端 case:完美分类器 AUC 应该是 1.0
scores_perfect = torch.tensor([0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2])
target_perfect = torch.tensor([1, 1, 1, 1, 0, 0, 0, 0])
print(f"\n完美分类器 AUC = {auc_mann_whitney(scores_perfect, target_perfect):.4f}  (应为 1.0)")

# 极端 case:随机分类器 AUC 应该 ≈ 0.5
torch.manual_seed(42)
scores_random = torch.rand(1000)
target_random = torch.randint(0, 2, (1000,))
print(f"随机分类器 AUC = {auc_mann_whitney(scores_random, target_random):.4f}  (应 ≈ 0.5)")


In [ ]:
import random
nums = random.sample(range(1, 100), 10)
print(nums)
def partition(i, j):
    val = nums[j]
    l, r = i, j
    while l < r:
        while l < r and nums[l] < val:
            l +=1
        while l < r and nums[r] > val:
            r -= 1
        nums[l], nums[r] = nums[r], nums[l]
        
    return l

print(partition(0, len(nums)-1))
nums

## 对比学习 / Contrastive Loss

### InfoNCE

InfoNCE 是对比学习的"原型"loss,理解它就能套到 SimCLR / MoCo / CLIP 上。

设 query $q$,1 个正样本 $k^+$,$N$ 个负样本 $\{k^-_i\}$:

$$
\mathcal{L}_{\text{InfoNCE}} = -\log \frac{\exp(\text{sim}(q, k^+) / \tau)}{\sum_{i \in \{+, 1..N\}} \exp(\text{sim}(q, k_i) / \tau)}
$$

- `sim` 通常是 **cosine similarity**(要先 L2-normalize,否则 inner product 会被范数污染)
- $\tau$ 是温度(常取 0.07~0.5),温度越小,模型越关注 hard negatives
- 本质就是 cross-entropy:logits = sim/τ,target = 正样本 index

### CLIP Loss

CLIP 把 InfoNCE 做成**对称**版本:image→text + text→image,各算一次 CE 取平均。
batch 内所有其他样本当成负样本(in-batch negatives)。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F




def info_nce(
    anchor: torch.Tensor,       # [B, D]
    positive: torch.Tensor,     # [B, D]
    negatives: torch.Tensor,    # [B, N, D]  或 [N, D](所有 anchor 共享)
    temperature: float = 0.07,
) -> torch.Tensor:
    """
    显式 anchor/positive/negatives 版本的 InfoNCE(单 batch)。
    """
    a = F.normalize(anchor, dim=-1)                       # [B, D]
    p = F.normalize(positive, dim=-1)                     # [B, D]
    n = F.normalize(negatives, dim=-1)                    # [B, N, D] or [N, D]

    # 正样本相似度: [B]
    pos_sim = (a * p).sum(dim=-1, keepdim=True) / temperature    # [B, 1]

    # 负样本相似度
    if n.dim() == 2:    # [N, D] -> 所有 anchor 共享
        neg_sim = a @ n.t() / temperature                          # [B, N]
    else:               # [B, N, D]
        # neg_sim = torch.einsum("bd,bnd->bn", a, n) / temperature   # [B, N]
        neg_sim = torch.bmm(a.unsqueeze(1), n.transpose(1, 2)).squeeze(1) / temperature  # [B, N]

    # logits = [pos | negs],target = 0(正样本在第一列)
    logits = torch.cat([pos_sim, neg_sim], dim=-1)                # [B, 1+N]
    target = torch.zeros(logits.size(0), dtype=torch.long, device=logits.device)
    return F.cross_entropy(logits, target)


# ============ 验证 ============
torch.manual_seed(0)
B, D, N = 4, 64, 7

anchor = torch.randn(B, D)
positive = anchor + 0.1 * torch.randn(B, D)         # 正样本:接近 anchor
negatives = torch.randn(B, N, D)                    # 负样本:随机

loss = info_nce(anchor, positive, negatives, temperature=0.1)
print(f"InfoNCE loss = {loss.item():.4f}   (≈ log(1+N) = {torch.log(torch.tensor(1.0 + N)):.4f} 在随机初始化时)")

# 对照:自己实现 CE 不用 F.cross_entropy
a = F.normalize(anchor, dim=-1)
p = F.normalize(positive, dim=-1)
n = F.normalize(negatives, dim=-1)
pos_sim = (a * p).sum(-1, keepdim=True) / 0.1
neg_sim = torch.einsum("bd,bnd->bn", a, n) / 0.1
logits = torch.cat([pos_sim, neg_sim], dim=-1)
manual = -logits[:, 0] + torch.logsumexp(logits, dim=-1)
assert torch.allclose(loss, manual.mean(), atol=1e-5)
print("✅ 公式与手写 -log p + logsumexp 一致")


In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def clip_loss(
    image_features: torch.Tensor,   # [B, D]   已经 L2-normalized 或在此处 normalize
    text_features: torch.Tensor,    # [B, D]
    temperature: float = 0.07,      # CLIP 论文学到的温度,常写成 log_scale 可学习参数
) -> torch.Tensor:
    """
    对称 CLIP loss = (CE(I→T) + CE(T→I)) / 2

    实现:
      1. L2 normalize
      2. logits = I @ T.T / τ         [B, B]
      3. target = arange(B)  (对角线是正样本对)
      4. CE 两个方向 + 平均
    """
    img = F.normalize(image_features, dim=-1)             # [B, D]
    txt = F.normalize(text_features, dim=-1)              # [B, D]

    logits = img @ txt.t() / temperature                  # [B, B]
    target = torch.arange(img.size(0), device=img.device)

    loss_i2t = F.cross_entropy(logits, target)            # 行向:每个 image 找对应 text
    loss_t2i = F.cross_entropy(logits.t(), target)        # 列向:每个 text 找对应 image
    return (loss_i2t + loss_t2i) / 2


class CLIPInfoNCE(nn.Module):
    """
    生产写法:温度作为可学习参数 (log_scale),与 CLIP 官方一致
    """
    def __init__(self, init_temperature: float = 0.07):
        super().__init__()
        # 数值稳定:存 log(1/τ),用 exp 还原
        self.log_scale = nn.Parameter(torch.tensor(math.log(1.0 / init_temperature)))

    def forward(self, image_features: torch.Tensor, text_features: torch.Tensor) -> torch.Tensor:
        img = F.normalize(image_features, dim=-1)
        txt = F.normalize(text_features, dim=-1)
        logits = img @ txt.t() * self.log_scale.exp()    # τ = 1 / exp(log_scale)
        target = torch.arange(img.size(0), device=img.device)
        return 0.5 * (F.cross_entropy(logits, target) + F.cross_entropy(logits.t(), target))


# ============ 验证 ============
torch.manual_seed(0)
B, D = 8, 128

# 完美匹配的情形:image == text,loss 应趋近 0
img_perfect = F.normalize(torch.randn(B, D), dim=-1)
loss_perfect = clip_loss(img_perfect, img_perfect.clone(), temperature=0.1)
print(f"完美匹配: loss = {loss_perfect.item():.4f}  (应趋近 0)")

# 完全随机的情形:batch=8,理论 loss ≈ log(8) = 2.079
img = torch.randn(B, D)
txt = torch.randn(B, D)
loss_random = clip_loss(img, txt, temperature=1.0)   # τ=1 让分布更平
print(f"随机匹配: loss = {loss_random.item():.4f}  (理论 ≈ log(B) = {math.log(B):.4f})")

# 可学习温度版
m = CLIPInfoNCE(init_temperature=0.1)
loss = m(img, txt)
print(f"CLIPInfoNCE (可学习温度): loss = {loss.item():.4f}, 当前 τ = {1 / m.log_scale.exp().item():.4f}")


## GEMM & Flash Attention

### 1) GEMM 手撕

面试常考的 `C = A @ B`,三种实现层次:
- **Naive**:`O(N³)` 三重循环,最朴素
- **Tiled / Blocked**:分块访问,提升 cache 命中(真正的"手撕 GEMM")
- **Vectorized**:向量化 + reshape,比朴素版快

### 2) Online Softmax (Flash Attention 核心 trick)

经典 softmax 需要 2 趟扫描:第一趟找 max,第二趟求和。
**Online softmax** 用流式更新公式,**1 趟**就能算出稳定的 softmax 归一化值:

$$
m_{\text{new}} = \max(m, x_i), \quad
s_{\text{new}} = s \cdot e^{m - m_{\text{new}}} + e^{x_i - m_{\text{new}}}
$$

每来一个新 token,就**重缩放之前的累加和**,再加新贡献。这是 Flash Attention 能在 SRAM 里做 tiling 而不需要全量 materialize 的根本原因。

In [ ]:
import torch
import torch.nn.functional as F


# ==================== 1. GEMM 三层实现 ====================

def gemm_naive(A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
    """最朴素三重循环,O(N³),纯 Python 慢但好懂"""
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    C = torch.zeros(M, N, dtype=A.dtype)
    for i in range(M):
        for j in range(N):
            acc = 0.0
            for k in range(K):
                acc += A[i, k] * B[k, j]
            C[i, j] = acc
    return C


def gemm_tiled(A: torch.Tensor, B: torch.Tensor, block: int = 32) -> torch.Tensor:
    """
    分块 GEMM:把内层 K 维切成 block,
    每次只把 A[:, k:k+block] @ B[k:k+block, :] 累加到 C。
    优点:
      1. cache 友好:小块数据可以装进 L1/L2
      2. 容易向量化:每块用 bmm 一次跑完
    这是面试手撕 GEMM 的"标准答案",再往下就是 SIMD/cuBLAS 级别
    """
    M, K = A.shape
    K2, N = B.shape
    assert K == K2
    C = torch.zeros(M, N, dtype=A.dtype)

    for k0 in range(0, K, block):
        # 一次处理 K 的一小段 [k0, k0+block)
        A_block = A[:, k0 : k0 + block]          # [M, block]
        B_block = B[k0 : k0 + block, :]          # [block, N]
        C += A_block @ B_block                   # 累加
    return C


# ==================== 2. Online Softmax ====================

def softmax_2pass(x: torch.Tensor):
    """经典两趟 softmax:先 max,再 exp+sum,最后归一化"""
    m = x.max(dim=-1, keepdim=True).values
    e = torch.exp(x - m)
    s = e.sum(dim=-1, keepdim=True)
    return e / s, m, s                          # 同时返回 m/s 用于 flash attention


def softmax_online(x: torch.Tensor, eps: float = 1e-12):
    """
    Online softmax:一趟扫描完成。
    对最后一维做流式更新,数学上与两趟版本完全等价。
    这个函数本身没有性能优势(向量化下两趟更快),价值在于:
      - 演示 trick 的等价性
      - Flash Attention 内部用它处理 tiled KV 块
    """
    *batch_dims, D = x.shape
    x_flat = x.reshape(-1, D)

    out = torch.empty_like(x_flat)
    for row in range(x_flat.size(0)):
        m = float("-inf")
        s = 0.0
        # 第一遍:流式更新 m 和 s
        for i in range(D):
            xi = x_flat[row, i].item()
            m_new = max(m, xi)
            s = s * torch.exp(torch.tensor(m - m_new)).item() + torch.exp(torch.tensor(xi - m_new)).item()
            m = m_new
        # 第二遍:用最终的 m 归一化
        out[row] = torch.exp(x_flat[row] - m) / s
    return out.reshape(x.shape)


def online_softmax_block(M_row: torch.Tensor, new_block: torch.Tensor,
                         m_old: torch.Tensor, s_old: torch.Tensor):
    """
    Flash Attention 内核里反复调用的"流式更新"算子。

    参数:
        M_row     : [B, heads, Q]   当前查询行对最新 KV 块的 score
        new_block : [B, heads, Q, K_block]  当前要并入的 score 块
        m_old     : [B, heads, Q, 1]  历史最大值
        s_old     : [B, heads, Q, 1]  历史分母和

    返回:
        m_new, s_new:  更新后的 running max / running sum
    (省略 P 块的 rescale,在 attention 里还有这一步)
    """
    m_block = new_block.max(dim=-1, keepdim=True).values           # 新块最大值
    m_new = torch.maximum(m_old, m_block)                          # 全局新 max
    # 把历史 s 缩放到新 max 下,再加新块的 sum
    s_old_rescaled = s_old * torch.exp(m_old - m_new)
    s_block = torch.exp(new_block - m_new).sum(dim=-1, keepdim=True)
    s_new = s_old_rescaled + s_block
    return m_new, s_new


# ==================== 验证 ====================
torch.manual_seed(0)

# GEMM
A = torch.randn(16, 32)
B = torch.randn(32, 24)

C_naive = gemm_naive(A, B)
C_tiled = gemm_tiled(A, B, block=8)
C_ref = A @ B
assert torch.allclose(C_naive, C_ref, atol=1e-4), "naive GEMM 不对"
assert torch.allclose(C_tiled, C_ref, atol=1e-4), "tiled GEMM 不对"
print(f"✅ GEMM: naive / tiled 与 torch.matmul 一致 (max_err="
      f"{(C_tiled - C_ref).abs().max().item():.2e})")

# Softmax 等价性
x = torch.randn(2, 8) * 5     # 乘 5 拉开数值,看稳定性
sm_2pass, _, _ = softmax_2pass(x)
sm_online = softmax_online(x)
assert torch.allclose(sm_2pass, sm_online, atol=1e-5), \
    f"online softmax 与两趟版不一致:\n{sm_2pass}\n{sm_online}"
print(f"✅ Online Softmax 与两趟版数值一致 (max_err="
      f"{(sm_2pass - sm_online).abs().max().item():.2e})")
print("   行和 = ", sm_online.sum(-1).tolist(), " (应全为 1)")


## 大模型后训练:DeepSpeed ZeRO 源码精讲

### 面试常考的 ZeRO 三阶段

| Stage | 切什么 | 通信量 | 显存节省 |
|-------|--------|--------|----------|
| ZeRO-1 | Optimizer states (Adam: m, v) | 与 baseline 相同 | ~4x |
| ZeRO-2 | + Gradients | Reduce-Scatter 替代 All-Reduce | ~8x |
| ZeRO-3 | + Parameters (权重本身) | 前向/反向各一次 All-Gather + Reduce-Scatter | ~N_gpu 倍 |

### 面试官最爱问的两个机制

1. **ZeRO-3 怎么"偷偷"把参数切走的?** → `deepspeed.zero.Init()` 上下文管理器**hook 住 `nn.Module.__init__`**,模块一被实例化就立即 partition。
2. **前向传播时全切片参数怎么用?** → `GatheredParameters` 上下文管理器触发 **all-gather**,临时拼回完整参数,算完立即释放。

下面是 DeepSpeed 源码 `deepspeed/runtime/zero/partition_parameters.py` 的极简复刻,展示 API 形态和核心逻辑。

In [ ]:
"""
DeepSpeed ZeRO-3 极简源码复刻(单进程模拟)

对应真实源码路径(deepspeed 0.14+):
  deepspeed/runtime/zero/partition_parameters.py  → Init / GatheredParameters
  deepspeed/runtime/zero/stage3.py                → 训练步的 hook
  deepspeed/runtime/engine.py                     → DeepSpeedEngine 顶层入口

设计要点(面试必答):
1. **怎么自动 partition?**
   DeepSpeed 用栈帧深度 `_external_init_level` 判断"最外层 __init__",
   在最外层返回时(所有子模块和参数都注册完)统一 partition。
   PyTorch 没有现成 hook,只能靠 `inspect.stack()` 数 frame。

2. **单进程模拟版怎么做最直观?**
   构造时给 Parameter 打 _to_partition 标志,
   在 Init 上下文退出前调用 `_partition_all(module)` 统一切片。
   等价于 DeepSpeed 在最外层 init 返回时做的事。
"""

import torch
import torch.nn as nn
import contextlib
from typing import Optional


# ==================== 1. 模拟分布式环境 ====================

class FakeProcessGroup:
    """单进程模拟多 GPU。真实环境用 torch.distributed"""
    def __init__(self, world_size: int, rank: int):
        self.world_size = world_size
        self.rank = rank

    def all_gather(self, shards: list[torch.Tensor]) -> torch.Tensor:
        """各 rank 的 shard 拼回完整 tensor (真实环境:all-gather 集合通信)"""
        return torch.cat(shards, dim=0)


PG = FakeProcessGroup(world_size=4, rank=0)


def _partition_parameter(param: nn.Parameter, pg: FakeProcessGroup):
    """把 param.data 切成 world_size 份,只留本 rank 的那一块"""
    if not param.requires_grad or getattr(param, "_partitioned", False):
        return
    data = param.data
    chunked = data.flatten().chunk(pg.world_size)         # 切 world_size 份
    param._all_shards = [c.detach().clone() for c in chunked]
    param.data = chunked[pg.rank].clone()                 # 本 rank 只持有自己的 view
    param._partitioned = True
    param._full_shape = data.shape


# ==================== 2. Init 上下文管理器 (ZeRO-3 核心 API) ====================

class ZeroInitContext:
    """
    对应 deepspeed.zero.Init(...)
    工作流程:
      进入 → 标记"接下来注册的 Parameter 都属于 zero-3 模型"
      退出 → 遍历整个 module 树,把所有 Parameter partition 掉
    (DeepSpeed 真实版在最外层 __init__ 返回时做,语义等价)
    """
    def __init__(self, enabled: bool = True):
        self.enabled = enabled

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        if not self.enabled:
            return
        # 退出时统一 partition — 等价于 DeepSpeed "最外层 init 返回" 时机
        # 注意:这里没真的 hook __setattr__,演示用直接遍历
        # 真实 DeepSpeed 用 _external_init_level 栈帧计数定位最外层
        _partition_all(_registered_model)


def _partition_all(module: nn.Module):
    """递归遍历所有子模块,partition 它们的可训练参数"""
    for p in module.parameters():
        _partition_parameter(p, PG)


_registered_model = None  # 全局占位,真实代码里 zero.Init 持有 model 引用


def register_for_zero(model: nn.Module):
    """配合 ZeroInitContext 使用的注册函数"""
    global _registered_model
    _registered_model = model


# ==================== 3. GatheredParameters (前向时拼回) ====================

@contextlib.contextmanager
def GatheredParameters(params, modifier_rank: Optional[int] = None):
    """进入时 all-gather 拼回完整 shape,退出时再 partition 回去"""
    gathered = []
    for p in params:
        if getattr(p, "_partitioned", False):
            full = PG.all_gather(p._all_shards).view(p._full_shape)
            p.data = full
            p._partitioned = False
            gathered.append(p)
    try:
        yield
    finally:
        for p in gathered:
            p._partitioned = False
            _partition_parameter(p, PG)


# ==================== 4. 演示 ====================

print(f"=== 模拟 ZeRO-3 in {PG.world_size} GPUs ===\n")

# 4.1 在 Init 上下文里构造模型,退出时统一 partition
with ZeroInitContext(enabled=True):
    model = nn.Sequential(
        nn.Linear(128, 256),
        nn.ReLU(),
        nn.Linear(256, 10),
    )
    register_for_zero(model)        # 真实 DeepSpeed 不需要这步,Init 内部直接拿 model

# 看看参数 shape
print(f"{'参数名':<15} {'完整 shape':<18} {'rank0 分片':<18} {'节省'}")
print("-" * 65)
for name, p in model.named_parameters():
    full = tuple(p._full_shape) if hasattr(p, "_full_shape") else tuple(p.shape)
    shard = tuple(p.shape)
    print(f"{name:<15} {str(full):<18} {str(shard):<18} ≈{PG.world_size}x")

# 4.2 前向时用 GatheredParameters 临时拼回
print("\n=== 前向传播 (GatheredParameters 自动 gather) ===")
x = torch.randn(8, 128)
with GatheredParameters(list(model.parameters())):
    print(f"  上下文内 linear.weight.shape = {tuple(model[0].weight.shape)}  (完整)")
    out = model(x)
    print(f"  output: {tuple(out.shape)}")
print(f"  退出后 linear.weight.shape = {tuple(model[0].weight.shape)}  (回到分片)")


# ==================== 5. 真实 DeepSpeed 用法对照 ====================
print("\n=== 真实 DeepSpeed 训练代码 ===")
print("""
import deepspeed

# 用 zero.Init 包住模型构造
with deepspeed.zero.Init(config_dict_or_path=ds_config):
    model = MyModel(...)

# 或者在 initialize 里自动套 Init
ds_config = {
    "zero_optimization": {
        "stage": 3,
        "param_persistence_threshold": 0,   # < N 参数的层不切,减少通信
        "overlap_comm": True,                # 计算/通信重叠
    },
    "train_micro_batch_size_per_gpu": 4,
    "optimizer": {"type": "AdamW", "params": {"lr": 1e-5}},
}
model, optimizer, _, _ = deepspeed.initialize(model=model, config=ds_config)

# 源码关键路径 (deepspeed 0.14):
#   runtime/engine.py:DeepSpeedEngine.__init__
#     ├─ _configure_zero()  根据 stage 创建 optimizer
#     │    └─ zero.Init() 上下文,hook Module.__init__ 自动 partition
#     └─ _initialize_optimizer_for_param_groups()
#
#   runtime/zero/partition_parameters.py:
#     ├─ ZeroInitContext.__enter__/exit__    _external_init_level 计数
#     ├─ _partition_param()                  按张量大小切到 world_size 份
#     └─ GatheredParameters                   前向/外部访问时 all-gather
#
#   runtime/zero/stage3.py:DeepSpeedZeroOptimizer_Stage3:
#     ├─ _pre_step()     all-gather 重建 params
#     ├─ _post_step()    重新 partition
#     └─ _zer3_grad_norms()  all-reduce 算 global grad norm
#
# 训练循环照常
for batch in dataloader:
    loss = model(batch).loss
    model.backward(loss)        # ← 自动 reduce-scatter gradients
    model.step()                # ← 自动 all-gather 更新后的参数
""")


## DeepSpeed / ZeRO 面试高频 Q&A

### Q1. ZeRO-1/2/3 各切什么?通信代价?显存节省?

| Stage | 切什么 | 显存节省 | 额外通信量(vs baseline) |
|-------|--------|----------|------------------------|
| baseline (DDP) | 啥都不切 | 1x | All-Reduce: `2Φ` (Φ = 参数量) |
| **ZeRO-1** | Optimizer states (Adam m/v) | ~4x | 不变(梯度仍 All-Reduce) |
| **ZeRO-2** | + Gradients | ~8x | Reduce-Scatter `Φ` + All-Gather `Φ` = `2Φ` (理论同 DDP) |
| **ZeRO-3** | + Parameters | ~`N` 倍 (N=gpu 数) | 前向 All-Gather `Φ` + 反向 All-Gather `Φ` + Reduce-Scatter `Φ` = `3Φ` |

**关键 insight**:ZeRO 的核心论点是 **"切得越多显存越好,但通信量同比增加"**,但**通信/计算 overlap** 后实际 wall time 几乎不变。

### Q2. ZeRO-3 vs PyTorch FSDP 区别?

| 维度 | DeepSpeed ZeRO-3 | PyTorch FSDP |
|------|------------------|--------------|
| 出现时间 | 2020 论文 | 2022 进入 torch.distributed |
| CPU/NVMe offload | ✅ ZeRO-Infinity | ❌(后续版本有) |
| 自动 hook Init | ✅ `zero.Init()` 上下文 | 需手动 `fully_shard(module)` |
| Mixed precision | 自家 `bf16 O2`/`O3` | 用 PyTorch AMP |
| Pipeline 并行 | ✅ DeepSpeed-Megatron | ❌ |
| 长上下文 | ✅ DeepSpeed-Ulysses | ❌ |
| 适用场景 | 训练超大模型(175B+) | 中型模型(7B-70B) |

**记忆口诀**:FSDP 是 ZeRO-3 的"PyTorch 原生精简版"。

### Q3. 通信原语:All-Reduce / Reduce-Scatter / All-Gather

| 原语 | 输入 | 输出 | 用途 |
|------|------|------|------|
| **All-Gather** | 每个 rank 有 1/N 数据 | 每个 rank 有全量 | ZeRO-3 前向拼回参数 |
| **Reduce-Scatter** | 每个 rank 有全量 | 每个 rank 有 1/N 的 sum | ZeRO-2/3 切分梯度 |
| **All-Reduce** | 每个 rank 有全量 | 每个 rank 有 sum 的全量 | DDP 同步梯度(= All-Gather + Reduce-Scatter) |

数学关系:`All-Reduce = Reduce-Scatter ⊕ All-Gather`,通信量 `2Φ`。所以 ZeRO-2 的"换成 Reduce-Scatter" 不是更省通信,而是**省显存**(只保留 1/N 梯度)。

### Q4. ZeRO-3 前向反向各需要哪些通信?

一个训练 step 的通信:
1. **Forward**:每层 `Linear` 前 `All-Gather` 拼回 weight → 算完即 `Free`
2. **Backward**:每层反向算 grad 前先 `All-Gather` weight,算完 grad 立即 `Reduce-Scatter` 把梯度切到各 rank
3. **Optimizer step**:每个 rank 只更新自己的 1/N 参数
4. 回到 1

**`overlap_comm=True`**:用单独 CUDA stream,前一层算的时候就开始 gather 下一层 → 把通信藏起来。

### Q5. `param_persistence_threshold` 是干啥的?

```python
"param_persistence_threshold": 100_000
```

小于这个元素数的参数**不切**(留在每个 rank 上完整副本)。
**原因**:小 tensor 通信开销 > 计算开销,切了反而慢。例如 LayerNorm 的 `weight` (几百元素) 不切,大 `Linear.weight` 切。

### Q6. ZeRO-Offload / ZeRO-Infinity 的区别?

- **ZeRO-Offload**(2021):把 optimizer states + gradients 卸到 **CPU 内存**,CPU 算 optimizer step,GPU 算 forward/backward。10GB GPU 训 10B 模型。
- **ZeRO-Infinity**(2021):在 Offload 基础上,**参数也切到 CPU + NVMe SSD**。打破 GPU 显存上限,40GB GPU 训 500B 模型。
- 代价:PCIe 带宽(32 GB/s)远低于 NVLink(600 GB/s),训练慢 1.5-3x。

### Q7. **LoRA + ZeRO-3** 的兼容坑(后训练高频问)

坑:**LoRA 的低秩 `A`/`B` 矩阵被 ZeRO-3 切了,反向时 gather 出错**。

解决方案:
1. `deepspeed.zero.Init()` 时给 LoRA 参数加 `enabled=False`(让 Init 跳过它们)
2. 用 `GatheredParameters` 上下文包住 LoRA 的前向
3. 或用 `deepspeed.runtime.zero.register_external_parameter` 显式声明

```python
with deepspeed.zero.Init(config_dict_or_path=ds_config, enabled=False):
    # LoRA 模块在外部构造,不进入 zero-3 partition
    lora = LoRALinear(...)
```

**PEFT 库**已经把这些坑处理了,用 `peft + deepspeed` 时正常 `prepare_model_for_kbit_training` 即可。

### Q8. DeepSpeed 和 Megatron-LM、Megatron-DeepSpeed 的关系?

- **Megatron-LM**(NVIDIA):张量并行(TP)+ 流水并行(PP)的鼻祖,擅长**单机多 GPU 内**的高速互联
- **DeepSpeed**(Microsoft):ZeRO 数据并行的鼻祖,擅长**跨机**扩展
- **Megatron-DeepSpeed**:两者结合,3D 并行 = DP(ZeRO-3) × TP × PP,训 175B+ 模型的标准方案
- 训 7B 以下:ZeRO-3 单独够用;训 70B+ :必须 3D 并行

### Q9. Pipeline Parallelism 在 DeepSpeed 里怎么实现?

把模型按层切成 K 个 stage,K 个 GPU 各跑一段,用 **micro-batching**(把一个 batch 切成 M 个 micro-batch)填满 pipeline bubble。
- **GPipe** 风格:所有 micro-batch forward 完再统一 backward(bubble 大)
- **PipeDream-Flush**(`1F1B`):forward 一个 micro 立即 backward 一个,bubble 小(DeepSpeed 默认)
- bubble 比例 ≈ `(K-1) / M`,所以 micro-batch 越多越高效

### Q10. **DeepSpeed-Ulysses**(2023,长序列注意力)

把 head 维度切开:每个 rank 只持有 `H/N` 个 head,通过 **AllToAll + AllReduce** 在序列维度上做切分。
- 传统 ZeRO 切的是 `[B, S, H]` 里的 batch B
- Ulysses 切的是 sequence S
- 训 100K+ 长上下文必备,Flash Attention v2 原生支持


In [ ]:
import torch
a = torch.tensor([1.0, 2.0, 3.0])
a.cumsum_(0)  # tensor([1., 3., 6.])
print(a)




In [ ]:
a = [1,2,3]

print(ord('A'))
